In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch
import sys
import os
sys.path.append(os.path.abspath(".."))
from core_ntsa.generators import simulate_lorenz, simulate_henon
from core_ntsa.noise_tools import add_white_noise, add_colored_noise
from core_ntsa.metrics import calculate_wolf_lle

lorenz_data = simulate_lorenz(t_span=50.0, dt=0.01)
clean_x = lorenz_data[0]

x_white_noise = add_white_noise(clean_x, snr_db=40.0)
x_clored_noise = add_colored_noise(clean_x, color='pink', snr_db=30.0)

In [8]:
# --- THIẾT LẬP THÔNG SỐ ĐỘNG LỰC HỌC CHO LORENZ ---
# dt = 0.01 (Đã lấy từ simulate_lorenz)
delay_tau = 15       # Độ trễ thời gian (Tránh các láng giềng quá gần trên cùng quỹ đạo)
m_embed = 3          # Chiều nhúng (Lorenz có 3 biến gốc, m=3 là tối thiểu)
evolve_steps = 20    # Số bước tiến hóa (tương đương 0.2s thực tế để quỹ đạo giãn ra đủ lớn)
max_angle = 0.3     # Dung sai góc (khoảng 17 độ) để bảo toàn hướng véc-tơ

print("Đang khởi chạy cỗ máy Wolf trên các môi trường nhiễu...\n")
print(f"{'Môi trường dữ liệu':<25} | {'LLE (1/s)':<15}")
print("-" * 45)

# 1. Chạy trên dữ liệu Sạch (Ground Truth)
lle_clean = calculate_wolf_lle(
    signal=clean_x, delay=delay_tau, m=m_embed, 
    evolve_steps=evolve_steps, dt=0.01, max_angle=max_angle
)
print(f"{'1. Lorenz Sạch lý tưởng':<25} | {lle_clean:<15.4f}")

# 2. Chạy trên dữ liệu Nhiễu Trắng (White Noise - SNR 40dB)
lle_white = calculate_wolf_lle(
    signal=x_white_noise, delay=delay_tau, m=m_embed, 
    evolve_steps=evolve_steps, dt=0.01, max_angle=max_angle
)
print(f"{'2. Nhiễu Trắng (SNR 40dB)':<25} | {lle_white:<15.4f}")

# 3. Chạy trên dữ liệu Nhiễu Hồng (Pink Noise - SNR 30dB)
lle_colored = calculate_wolf_lle(
    signal=x_clored_noise, delay=delay_tau, m=m_embed, 
    evolve_steps=evolve_steps, dt=0.01, max_angle=max_angle
)
print(f"{'3. Nhiễu Hồng (SNR 30dB)':<25} | {lle_colored:<15.4f}")
print("-" * 45)

Đang khởi chạy cỗ máy Wolf trên các môi trường nhiễu...

Môi trường dữ liệu        | LLE (1/s)      
---------------------------------------------
1. Lorenz Sạch lý tưởng   | 0.8238         
2. Nhiễu Trắng (SNR 40dB) | 1.0348         
3. Nhiễu Hồng (SNR 30dB)  | 1.6973         
---------------------------------------------


### Phân Tích Thực Nghiệm: Tử Huyệt Của Thuật Toán Wolf Trước Nhiễu

> **Nhận xét cốt lõi:** Thuật toán Wolf bộc lộ yếu điểm chí mạng khi đối mặt với dữ liệu nhiễu. Khác với cơ chế "trung bình bầy đàn" của Kantz, việc bám theo một quỹ đạo đơn lẻ khiến cỗ máy Wolf dễ bị đánh lừa bởi các dao động giả tạo, làm thổi phồng tốc độ phân kỳ thực tế.

---

#### 1. Lorenz Sạch (0.8238): Giới hạn của kẻ đơn độc
Con số **0.8238** là một kết quả rất tốt và bám sát giá trị lý thuyết ~**0.906**. Sự thiếu hụt một chút so với lý thuyết xuất phát từ cơ chế hình học của thuật toán:
* Trong không gian 3D hữu hạn, rất hiếm khi thuật toán Wolf tìm được một láng giềng mới có góc lệch bằng đúng **0 độ** so với vector cũ. 
* Mỗi lần thực hiện thay thế láng giềng với một góc lệch nhỏ, thuật toán vô tình đánh mất một phần tốc độ giãn nở thực sự của trục chính.

#### 2. Nhiễu Trắng ở 40dB (1.0348): Sự mù lòa cục bộ
Với mức tín hiệu trên nhiễu (SNR) là **40dB** (nhiễu cực kỳ nhỏ, chỉ bằng **1%** công suất tín hiệu), thế nhưng giá trị LLE đã bị thổi phồng lên hơn **25%**. 
* Nhiễu trắng tạo ra các bước nhảy zig-zag siêu nhỏ trên quỹ đạo hệ thống. 
* Khi thuật toán Wolf đi tìm láng giềng mới, các vector bị bẻ cong ngẫu nhiên khiến điều kiện "bảo toàn góc" bị phá vỡ hoàn toàn. 
* **Hậu quả:** Thuật toán liên tục thay thế nhầm láng giềng, biên dịch sai sự nảy bật ngẫu nhiên của nhiễu thành động lực học hỗn loạn nội tại.

#### 3. Nhiễu Hồng ở 30dB (1.6973): Đòn chí mạng
Đây là phát hiện thú vị nhất và cũng là minh chứng rõ nhất cho sự thất bại của thuật toán khi thiếu cơ chế khử nhiễu. Nhiễu hồng (Pink noise hay 1/f noise) không dao động ngẫu nhiên như nhiễu trắng, mà có "quán tính" với các dải sóng chậm trôi dạt.
* Thuật toán Wolf bị đánh lừa hoàn toàn bởi đặc tính này. 
* Nó ngoan ngoãn bám theo các sóng trôi dạt của nhiễu hồng và tưởng nhầm đó là sự phân kỳ của chính hệ thống Lorenz. 
* **Hậu quả:** Kết quả LLE bị bơm phồng lên gấp đôi (**1.6973**), hoàn toàn làm mất đi giá trị phân tích định lượng ban đầu.